# Download das bases CNPJ — Receita Federal (2026-08)

Este notebook **somente baixa** os arquivos necessários para a análise de Piracicaba:
- `Cnaes.zip`
- `Municipios.zip`
- `Motivos.zip`
- `Estabelecimentos0.zip` até `Estabelecimentos9.zip`

Arquivos já existentes na pasta de destino são ignorados.

In [ ]:
from pathlib import Path
import requests
from tqdm.auto import tqdm

# Pasta onde os arquivos serão salvos
PASTA_DESTINO = Path(
    r"C:\Users\ielde\OneDrive\Área de Trabalho\GABRIEL\e-commerce\bases\_cnpj\bases\_cnpj\_receita-federal\_202608"
)

PASTA_DESTINO.mkdir(parents=True, exist_ok=True)

print("Destino:")
print(PASTA_DESTINO)

In [ ]:
# URL-base da competência 2026-08
BASE_URL = "https://arquivos.receitafederal.gov.br/dados/cnpj/dados_abertos_cnpj/2026-08/"

ARQUIVOS = [
    "Cnaes.zip",
    "Municipios.zip",
    "Motivos.zip",
] + [
    f"Estabelecimentos{i}.zip"
    for i in range(10)
]

ARQUIVOS

In [ ]:
def baixar_arquivo(url, destino, chunk_size=1024 * 1024):
    """
    Baixa um arquivo com barra de progresso.
    Se o arquivo já existir e tiver tamanho > 0, ele será ignorado.
    """
    destino = Path(destino)

    if destino.exists() and destino.stat().st_size > 0:
        print(f"✓ Já existe: {destino.name}")
        return

    arquivo_parcial = destino.with_suffix(destino.suffix + ".part")

    headers = {}
    modo = "wb"
    baixado = 0

    # Permite continuar um download interrompido, se houver .part
    if arquivo_parcial.exists():
        baixado = arquivo_parcial.stat().st_size
        headers["Range"] = f"bytes={baixado}-"
        modo = "ab"

    with requests.get(
        url,
        stream=True,
        timeout=300,
        headers=headers
    ) as response:
        response.raise_for_status()

        # Se o servidor não aceitou Range, recomeça do zero
        if baixado > 0 and response.status_code != 206:
            baixado = 0
            modo = "wb"

        tamanho_restante = int(
            response.headers.get("content-length", 0)
        )

        tamanho_total = baixado + tamanho_restante

        with open(arquivo_parcial, modo) as f, tqdm(
            total=tamanho_total if tamanho_total > 0 else None,
            initial=baixado,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=destino.name
        ) as barra:

            for bloco in response.iter_content(chunk_size=chunk_size):
                if bloco:
                    f.write(bloco)
                    barra.update(len(bloco))

    arquivo_parcial.replace(destino)
    print(f"✓ Concluído: {destino.name}")

In [ ]:
for nome_arquivo in ARQUIVOS:
    url = BASE_URL + nome_arquivo
    destino = PASTA_DESTINO / nome_arquivo

    try:
        baixar_arquivo(url, destino)
    except Exception as e:
        print(f"✗ Erro em {nome_arquivo}: {e}")

In [ ]:
print("\nArquivos encontrados na pasta:\n")

for nome_arquivo in ARQUIVOS:
    caminho = PASTA_DESTINO / nome_arquivo

    if caminho.exists():
        tamanho_gb = caminho.stat().st_size / (1024 ** 3)
        print(f"✓ {nome_arquivo:<24} {tamanho_gb:.2f} GB")
    else:
        print(f"✗ {nome_arquivo:<24} NÃO ENCONTRADO")